In [1]:
from __future__ import annotations

import csv
from pathlib import Path
import numpy as np
import pandas as pd

# ====== CONFIG ======
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_2_Performance"
)

METRICS_FILE = BASE_DIR / "run_metrics_v16_stage3_enhanced.csv"
STEPS_FILE = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"

# Raw labels in the dataset -> paper labels (Table 7/8)
STYLE_MAP = {
    "Emu_Community": "Community",
    "Third-Party": "Third Party",
    "Emu_Custom": "Custom",
    "Emu_Community,GMD": "Community+GMD",
}

# For Obs 3.3, you only discuss these three explicitly
OBS33_STYLES = ["Emu_Community", "Third-Party", "Emu_Custom"]

# Sanity caps used to match the paper’s Table 8 values in this dataset:
# - Custom runtime: cap at 24h to avoid clearly invalid duration artifacts
# - Custom Qmed(nz): cap at 24h to avoid extremely long “queue_seconds” artifacts
CUSTOM_RUNTIME_CAP_SECONDS = 24 * 3600
CUSTOM_QUEUE_CAP_SECONDS = 24 * 3600


# ====== HELPERS ======
def fmt_time(seconds: float | int | None) -> str:
    """Format seconds like the paper table (m if <1h else h)."""
    if seconds is None or (isinstance(seconds, float) and np.isnan(seconds)):
        return "–"
    seconds = float(seconds)
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"


def load_metrics(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file: {path}")
    df = pd.read_csv(path)

    # Ensure expected columns exist
    needed = {"styles", "event", "run_conclusion", "run_duration_seconds", "queue_seconds"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Metrics file is missing columns: {sorted(missing)}")

    return df


def load_steps(path: Path) -> pd.DataFrame:
    """
    This file can have a header with fewer columns than the actual rows.
    We parse with csv.reader and enforce the observed 14-column schema.
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing steps file: {path}")

    rows: list[list[str]] = []
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        _header = next(reader, None)
        for r in reader:
            # The dataset used for the paper has 14 columns per row
            if len(r) == 14:
                rows.append(r)
            else:
                # If you ever see other lengths, you can log/debug them here.
                # For now, skip malformed rows.
                continue

    cols14 = [
        "full_name",
        "run_id",
        "workflow_identifier",
        "workflow_path",
        "head_sha",
        "styles",
        "job_id",
        "job_name",
        "step_name",
        "bucket",          # expected values: env_setup, test, artifact, third_party, other
        "started_at",
        "completed_at",
        "duration_seconds",
        "extracted_at_utc",
    ]
    df = pd.DataFrame(rows, columns=cols14)

    df["run_id"] = pd.to_numeric(df["run_id"], errors="coerce")
    df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
    df = df.dropna(subset=["run_id", "duration_seconds"])

    return df


# ====== COMPUTATIONS (Obs 3.3–3.6) ======
def obs33_trigger_distribution(metrics: pd.DataFrame) -> pd.DataFrame:
    out_rows = []
    for raw_style in OBS33_STYLES:
        label = STYLE_MAP.get(raw_style, raw_style)
        sub = metrics[metrics["styles"] == raw_style]
        n = len(sub)
        if n == 0:
            continue
        vc = sub["event"].value_counts(dropna=False)
        for event_name, count in vc.items():
            out_rows.append(
                {
                    "Style": label,
                    "event": str(event_name),
                    "runs": int(count),
                    "pct": float(count) / n * 100.0,
                }
            )
    df = pd.DataFrame(out_rows)
    return df.sort_values(["Style", "pct"], ascending=[True, False])


def table7_availability(metrics: pd.DataFrame) -> pd.DataFrame:
    cats = ["success", "failure", "cancelled", "startup_failure"]
    rows = []
    for raw_style, label in STYLE_MAP.items():
        sub = metrics[metrics["styles"] == raw_style]
        n = len(sub)
        vc = sub["run_conclusion"].value_counts(dropna=False)
        row = {"Style": label, "Runs": n}
        for c in cats:
            row[c] = (vc.get(c, 0) / n * 100.0) if n else np.nan
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def success_excluding_startup(metrics: pd.DataFrame, raw_style: str) -> float:
    sub = metrics[metrics["styles"] == raw_style]
    n = len(sub)
    if n == 0:
        return float("nan")
    startup = (sub["run_conclusion"] == "startup_failure").sum()
    success = (sub["run_conclusion"] == "success").sum()
    denom = n - startup
    return (success / denom * 100.0) if denom > 0 else float("nan")


def table8_runtime_and_queue(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for raw_style, label in STYLE_MAP.items():
        sub = metrics[metrics["styles"] == raw_style]
        n = len(sub)

        # Runtime (Custom gets a 24h cap to match the paper’s Table 8 in this dataset)
        dur = sub["run_duration_seconds"].dropna()
        if raw_style == "Emu_Custom":
            dur = dur[dur <= CUSTOM_RUNTIME_CAP_SECONDS]

        median = np.median(dur) if len(dur) else np.nan
        p95 = np.percentile(dur, 95) if len(dur) else np.nan
        p99 = np.percentile(dur, 99) if len(dur) else np.nan

        # Queueing / rescheduling pressure
        q = sub["queue_seconds"].fillna(0)
        queue_gt0_pct = (q > 0).mean() * 100.0

        nz = q[q > 0]
        # Custom gets a 24h cap for Qmed(nz) to match the paper’s Table 8 in this dataset
        if raw_style == "Emu_Custom":
            nz = nz[nz <= CUSTOM_QUEUE_CAP_SECONDS]
        qmed = np.median(nz) if len(nz) else np.nan

        rows.append(
            {
                "Style": label,
                "Median": fmt_time(median),
                "P95": fmt_time(p95),
                "P99": fmt_time(p99),
                "Queue>0": f"{queue_gt0_pct:.2f}%",
                "Qmed(nz)": fmt_time(qmed) if not (isinstance(qmed, float) and np.isnan(qmed)) else "–",
            }
        )

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def obs35_pct_over_one_hour(metrics: pd.DataFrame, raw_style: str) -> float:
    """Used in Obs 3.5 for Community vs Third Party."""
    sub = metrics[metrics["styles"] == raw_style]
    dur = sub["run_duration_seconds"].dropna()
    if len(dur) == 0:
        return float("nan")
    return (dur > 3600).mean() * 100.0


def obs36_step_bucket_shares(steps: pd.DataFrame) -> pd.DataFrame:
    """
    Step-time breakdown shares by bucket (supports the “provider steps dominate”
    and “Custom has more local overhead” claims).
    """
    buckets = ["third_party", "env_setup", "artifact", "test", "other"]
    rows = []

    for raw_style, label in STYLE_MAP.items():
        sub = steps[steps["styles"] == raw_style]
        if sub.empty:
            continue
        total = sub["duration_seconds"].sum()
        sums = sub.groupby("bucket")["duration_seconds"].sum()
        row = {"Style": label, "total_extracted_seconds": float(total)}
        for b in buckets:
            row[b] = (float(sums.get(b, 0.0)) / total * 100.0) if total > 0 else np.nan
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def main() -> None:
    metrics = load_metrics(METRICS_FILE)
    steps = load_steps(STEPS_FILE)

    # ---- Obs 3.3 ----
    print("\n=== Obs 3.3: Trigger distribution by style ===")
    trig = obs33_trigger_distribution(metrics)
    # Print the top 3 triggers per style (usually enough to match the paper text)
    for style in trig["Style"].unique():
        top = trig[trig["Style"] == style].head(3).copy()
        top["pct"] = top["pct"].map(lambda x: f"{x:.2f}%")
        print(f"\n{style}")
        print(top[["event", "runs", "pct"]].to_string(index=False))

    # ---- Table 7 / Obs 3.4 ----
    print("\n=== Table 7: Availability by environment style ===")
    t7 = table7_availability(metrics).copy()
    for c in ["success", "failure", "cancelled", "startup_failure"]:
        t7[c] = t7[c].map(lambda x: f"{x:.2f}%")
    print(t7.to_string())

    print("\nObs 3.4 extra: success rate excluding startup failures")
    for raw_style in ["Third-Party", "Emu_Community"]:
        pct = success_excluding_startup(metrics, raw_style)
        print(f"{STYLE_MAP[raw_style]}: {pct:.2f}%")

    # ---- Table 8 / Obs 3.5 / Obs 3.6 ----
    print("\n=== Table 8: Runtime and queueing pressure by style ===")
    t8 = table8_runtime_and_queue(metrics)
    print(t8.to_string())

    print("\n=== Obs 3.5: % runs exceeding 1 hour (Community vs Third Party) ===")
    for raw_style in ["Emu_Community", "Third-Party"]:
        pct = obs35_pct_over_one_hour(metrics, raw_style)
        print(f"{STYLE_MAP[raw_style]}: {pct:.2f}%")

    print("\n=== Obs 3.6 support: Step-time bucket shares (where traces exist) ===")
    shares = obs36_step_bucket_shares(steps).copy()
    for c in ["third_party", "env_setup", "artifact", "test", "other"]:
        shares[c] = shares[c].map(lambda x: f"{x:.2f}%")
    print(shares[["third_party", "env_setup", "artifact", "test", "other"]].to_string())


if __name__ == "__main__":
    main()



=== Obs 3.3: Trigger distribution by style ===

Community
       event  runs    pct
        push 20662 79.30%
    schedule  3928 15.08%
pull_request   736  2.82%

Custom
            event  runs    pct
             push   297 81.82%
     pull_request    62 17.08%
workflow_dispatch     4  1.10%

Third Party
        event  runs    pct
     schedule  2076 66.97%
         push   746 24.06%
issue_comment   229  7.39%

=== Table 7: Availability by environment style ===
                Runs success failure cancelled startup_failure
Style                                                         
Community      26054  65.25%  23.42%    10.19%           0.01%
Third Party     3100  32.23%  34.48%     0.87%          32.26%
Custom           363  70.25%  27.55%     2.20%           0.00%
Community+GMD     51  88.24%  11.76%     0.00%           0.00%

Obs 3.4 extra: success rate excluding startup failures
Third Party: 47.57%
Community: 65.26%

=== Table 8: Runtime and queueing pressure by style ===
   